<div style="background-color: black;">
    <hr style="border: 4px solid skyblue;">
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 20px; font-family: TimesNewRoman; color: skyblue;">
    TIME SERIES DATA PROCESSING
    <br>
    NET TRANSFER CAPACITIES
</div>
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    Main Formatting Notebook
    <br>
    from PYPSA to DISPA-SET
</div>
<br>
<div style="text-align: justify; margin-left: 0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
This script is used to process the raw time series data for country interconecction capacities obtained from the PyPSA model simulations conducted for the Dispa-SET Unleashed project.
    <br>
The explanation text cells detail the entire process and help the reader understand how the final results were achieved step-by-step.
</div>
<br>
<hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    1. Notebook Set Up
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Importing needed libraries.
<hr style="border: 1px solid skyblue;">
</div>
</div>

In [72]:
import os
import csv
from datetime import datetime
import requests
import pandas as pd
from shutil import move
import numpy as np
import shutil
from bs4 import BeautifulSoup
import re
import io
import plotly.graph_objects as go
from typing import List, Dict, Tuple, Optional, Tuple
import re
from IPython.display import HTML
from difflib import get_close_matches
from collections import defaultdict
import warnings
from pathlib import Path
import glob
from itertools import permutations

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Auxiliar Code
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cell has the purpose to create the correponding folders with the name of all the EU countries available in the ENTSOE data base.
    <br>
    Uncomment it to use it just if needed.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [73]:
'''
# List of countries with their acronyms in parentheses
countries = [
    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",
    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",
    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",
    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",
    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",
    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",
    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",
    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",
    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",
    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slovenia   (SI)",
    "Spain         (ES)", "Sweden         (SE)", "Switzerland      (CH)", "Turkey     (TR)",
    "Ukraine       (UA)", "United Kingdom (UK)"
]

# Set the path where you want to create the folders
base_path = '/home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/HydroData/ScaledInflows'

# Ensure the base path exists
os.makedirs(base_path, exist_ok=True)

# Loop through the list of countries
for country_string in countries:
    # Use a regular expression to find the acronym inside the parentheses
    match = re.search(r'\((.*?)\)', country_string)

    # If a match is found, extract the acronym
    if match:
        acronym = match.group(1).strip()  # Use strip() to remove any extra whitespace
        folder_path = os.path.join(base_path, acronym)

        # Check if the folder already exists to avoid errors
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"Created folder: {folder_path}")
        else:
            print(f"Folder already exists: {folder_path}")
    else:
        print(f"Could not extract acronym from: {country_string}")

print("\nAll folders created successfully!")
'''

<>:1: SyntaxWarning: invalid escape sequence '\('
<>:1: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_2152992/2889290989.py:1: SyntaxWarning: invalid escape sequence '\('
  '''


'\n# List of countries with their acronyms in parentheses\ncountries = [\n    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",\n    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",\n    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",\n    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",\n    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",\n    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",\n    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",\n    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",\n    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",\n    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slove

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    2. Dispa-SET_Unleash Folder Path
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Determinning dynamically the zone_folder_path based on the location of the "Dispa-SET_Unleash" folder relative to the current working directory. 
<br>
    If the "Dispa-SET_Unleash" folder is copied to a different machine or location, the dispaSET_unleash_folder_path variable will automatically adjust accordingly.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [74]:
# 1. ------------------------------------------------------------------------------  Get the current working directory 
current_directory = os.getcwd()

# 2. --------------------------------------------------------  Navigate to the parent directory of "Dispa-SET_Unleash" 
dispaSET_unleash_parent_directory = os.path.dirname(current_directory)

# 3. ------------------------------------------------------------------ Get the path to the "Dispa-SET_Unleash" folder 
dispaSET_unleash_folder_path = os.path.dirname(dispaSET_unleash_parent_directory)

# 4. ------------------------------------------------------------- Construct the dispaSET_unleash_folder_name variable 
dispaSET_unleash_folder_name = os.path.basename(dispaSET_unleash_folder_path)

# 5. ------------------------------------------------------------------ ----------------------------------------- Done 
print("dispaSET_unleash_folder_name:", dispaSET_unleash_folder_name)
print("dispaSET_unleash_folder_path:", dispaSET_unleash_folder_path)

dispaSET_unleash_folder_name: Dispa-SET_Unleash
dispaSET_unleash_folder_path: /home/ray/Dispa-SET_Unleash


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.1. PyPSA Source Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
There are two sccenarios as source of PyPSA power plants raw data.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
Reference_Scenario
<li>
Suficiency_Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The scenario variable must be selected before proceeding to the next processing steps.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [75]:
# 1. ----------------------------------------------------------------------------------------- Set the needed scenatio 
#pypsa_scenario = "Reference_Scenario"
pypsa_scenario = "Suficiency_Scenario"

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("PyPSA Chosen Scenario:", pypsa_scenario)

PyPSA Chosen Scenario: Suficiency_Scenario


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.2. Dispa-SET Time Step
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The time series data must be resampled to a predetermined time step.
<br>
The UNLEASH project utilizes three levels of granularity:
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
One hour (1h) 
<li>
Thirty minutes (30min)
<li>
Fifteen minutes (15min)
</li>
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [76]:
# 1. ---------------------------------------------------------------- Set the time step to which data is formating to: 
data_target_time_step = '1h'
# data_target_time_step = '15min'
# data_target_time_step = '30min'

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("Chosen time step:", data_target_time_step)

1h


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.3. Secondary Folder Paths
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The many subfolders within the Unleash directory require their location paths to be defined for correct access during processing.
<br>
All of these are dependent on the chosen scenario.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [107]:
# 1. ------------------------------------- Get the path to the "Dispa-SET_Unleash_NTC" folder of the data used as base 
additional_path_1 = os.path.join("/Database/NTC", data_target_time_step)
ntc_base_data_folder_path = dispaSET_unleash_folder_path + additional_path_1

# 2. ----------------------------------------- Construct the Dispa-SET_NTC_name variable of the data used as reference 
ntc_base_data_folder_name = os.path.basename(ntc_base_data_folder_path)

# 3. ------------------------------------------------------------------------------------------------------------ Done
print("ntc_base_data_folder_name:", ntc_base_data_folder_name)
print("ntc_base_data_folder_path:", ntc_base_data_folder_path)
# ===========================================================================================================================================

# 1. ------------------------------------------------ Get the path to the "Dispa-SET_NTC" folder of the PyPSA raw data 
additional_path_2 = os.path.join("RawData_PyPSA", pypsa_scenario, "NTC")
ntc_pypsa_raw_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_2

additional_path_3 = os.path.join("RawData/NTC/Raw_Data_Sources")
ntc_matrix_raw_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_3

# 2. ---------------------------------- Construct the Dispa-SET_Unleash_NTC_folder_name variable of the PyPSA raw data 
ntc_pypsa_raw_data_folder_name = os.path.basename(ntc_pypsa_raw_data_folder_path)

ntc_matrix_raw_data_folder_name = os.path.basename(ntc_matrix_raw_data_folder_path)

# 3. ------------------------------------------------------------------------------------------------------------ Done
print("ntc_pypsa_raw_data_folder_name:", ntc_pypsa_raw_data_folder_name)
print("ntc_pypsa_raw_data_folder_path:", ntc_pypsa_raw_data_folder_path)

print("ntc_matrix_raw_data_folder_name:", ntc_matrix_raw_data_folder_name)
print("ntc_matrix_raw_data_folder_path:", ntc_matrix_raw_data_folder_path)
# ===========================================================================================================================================

# 1. --------------------------------- Get the path to the "Dispa-SET_NTC" folder of the PyPSA formated data 
additional_path_4 = os.path.join("Database_PyPSA", pypsa_scenario, "NTC", data_target_time_step)
ntc_pypsa_formated_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_4

# 2. ------------------- Construct the Dispa-SET_Unleash_Scalled_Inflows_folder_name variable of the PyPSA formated data 
ntc_pypsa_formated_data_folder_name = os.path.basename(ntc_pypsa_formated_data_folder_path)

# 3. -------------------------------------------------------------------------------------------------------------- Done
print("ntc_pypsa_formated_data_folder_name:", ntc_pypsa_formated_data_folder_name)
print("ntc_pypsa_formated_data_folder_path:", ntc_pypsa_formated_data_folder_path)

ntc_base_data_folder_name: 1h
ntc_base_data_folder_path: /home/ray/Dispa-SET_Unleash/Database/NTC/1h
ntc_pypsa_raw_data_folder_name: NTC
ntc_pypsa_raw_data_folder_path: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/NTC
ntc_matrix_raw_data_folder_name: Raw_Data_Sources
ntc_matrix_raw_data_folder_path: /home/ray/Dispa-SET_Unleash/RawData/NTC/Raw_Data_Sources
ntc_pypsa_formated_data_folder_name: 1h
ntc_pypsa_formated_data_folder_path: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/NTC/1h


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    3. Zone(s) Creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Entering the zone name or names (comment those ones that are not available data) where all data related to the corresponding zone are going to be storage.
<br>
For European country names use the ISO 3166-1 standars i.e. AT, BE, BG, CH.... etc. to give the zone_name.
</div>
<hr style="border: 1px solid skyblue;">

In [115]:
# 1. -------------------------------------------------------------------- Set the list of folder names to be addressed 
zone_names = [
                #"AL",
                #"AM",
                #"AT",
                #"AZ",
                #"BY",
                "BE",
                #"BA",
                #"BG",
                #"HR",
                #"CY",
                #"CZ",
                #"DK",
                #"EE",
                #"FI",
                "FR",
                #"GE",
                "DE",
                #"EL",
                #"HU",
                #"IS",
                #"IE",
                #"IT",
                #"XK",
                #"LV",
                #"LT",
                #"LU",
                #"MT",
                #"MD",
                #"ME",
                "NL",
                #"MK",
                #"NO",
                #"PL",
                #"PT",
                #"RO",
                #"RU",
                #"RS",
                #"SK",
                #"SI",
                #"ES",
                #"SE",
                #"CH",
                #"TR",
                #"UA",
                "UK"
             ]

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("Selected Zones:", zone_names)

Selected Zones: ['BE', 'FR', 'DE', 'NL', 'UK']


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The following dictionary specifies the possible alternative names—or synonyms/aliases—used in international nomenclature for the EU countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [112]:
# 1. --------------------------------------------------------- Set the list of the alternative acronyms per EU country 
zone_names_equivalences_dict = {

"AL"  :  {"Acronym": ["  "       ] ,   "name": ["Albania   "       ]},  
"AM"  :  {"Acronym": ["  "       ] ,   "name": ["Armenia   "       ]},
"AT"  :  {"Acronym": ["  "       ] ,   "name": ["Austria   "       ]},
"AZ"  :  {"Acronym": ["  "       ] ,   "name": ["Azerbaijan"       ]},
"BY"  :  {"Acronym": ["  "       ] ,   "name": ["Belarus"          ]},
"BE"  :  {"Acronym": ["  "       ] ,   "name": ["Belgium"          ]},
"BA"  :  {"Acronym": ["  "       ] ,   "name": ["Bosnia and Herz." ]},
"BG"  :  {"Acronym": ["  "       ] ,   "name": ["Bulgaria"         ]},
"HR"  :  {"Acronym": ["  "       ] ,   "name": ["Croatia"          ]},
"CY"  :  {"Acronym": ["  "       ] ,   "name": ["Cyprus"           ]},
"CZ"  :  {"Acronym": ["  "       ] ,   "name": ["Czech Republic"   ]},
"DK"  :  {"Acronym": ["  "       ] ,   "name": ["Denmark"          ]},
"EE"  :  {"Acronym": ["  "       ] ,   "name": ["Estonia"          ]},
"FI"  :  {"Acronym": ["  "       ] ,   "name": ["Finland"          ]},
"FR"  :  {"Acronym": ["  "       ] ,   "name": ["France"           ]},
"GE"  :  {"Acronym": ["  "       ] ,   "name": ["Georgia"          ]}, 
"DE"  :  {"Acronym": ["  "       ] ,   "name": ["Germany"          ]},
"EL"  :  {"Acronym": ["GR"       ] ,   "name": ["Greece"           ]},
"HU"  :  {"Acronym": ["  "       ] ,   "name": ["Hungary"          ]},
"IS"  :  {"Acronym": ["  "       ] ,   "name": ["Iceland"          ]},
"IE"  :  {"Acronym": ["  "       ] ,   "name": ["Ireland"          ]},
"IT"  :  {"Acronym": ["  "       ] ,   "name": ["Italy"            ]},
"XK"  :  {"Acronym": ["  "       ] ,   "name": ["Kosovo"           ]},
"LV"  :  {"Acronym": ["  "       ] ,   "name": ["Latvia"           ]},
"LT"  :  {"Acronym": ["  "       ] ,   "name": ["Lithuania"        ]},
"LU"  :  {"Acronym": ["  "       ] ,   "name": ["Luxembourg"       ]},
"MT"  :  {"Acronym": ["  "       ] ,   "name": ["Malta"            ]},
"MD"  :  {"Acronym": ["  "       ] ,   "name": ["Moldova"          ]},
"ME"  :  {"Acronym": ["  "       ] ,   "name": ["Montenegro"       ]},
"NL"  :  {"Acronym": ["  "       ] ,   "name": ["Netherlands"      ]},
"MK"  :  {"Acronym": ["  "       ] ,   "name": ["North Macedonia"  ]},
"NO"  :  {"Acronym": ["  "       ] ,   "name": ["Norway"           ]},
"PL"  :  {"Acronym": ["  "       ] ,   "name": ["Poland"           ]},
"PT"  :  {"Acronym": ["  "       ] ,   "name": ["Portugal"         ]},
"RO"  :  {"Acronym": ["  "       ] ,   "name": ["Romania"          ]},
"RU"  :  {"Acronym": ["  "       ] ,   "name": ["Russia"           ]},
"RS"  :  {"Acronym": ["  "       ] ,   "name": ["Serbia"           ]},
"SK"  :  {"Acronym": ["  "       ] ,   "name": ["Slovakia"         ]},
"SI"  :  {"Acronym": ["  "       ] ,   "name": ["Slovenia"         ]},
"ES"  :  {"Acronym": ["  "       ] ,   "name": ["Spain"            ]},
"SE"  :  {"Acronym": ["  "       ] ,   "name": ["Sweden"           ]},
"CH"  :  {"Acronym": ["  "       ] ,   "name": ["Switzerland"      ]},
"TR"  :  {"Acronym": ["  "       ] ,   "name": ["Turkey"           ]},
"UA"  :  {"Acronym": ["  "       ] ,   "name": ["Ukraine"          ]},
"UK"  :  {"Acronym": ["GB"       ] ,   "name": ["United Kingdom"   ]},
       
}

# 2. ------------------------------------------------ Create a new dictionary with only the keys present in zone_names
selected_zone_names_equivalences_dict = {
    key: zone_names_equivalences_dict[key]
    for key in zone_names
    if key in zone_names_equivalences_dict
}

# 3. ------------------------------------------------------------------------------------------------------------ Done 
for key, value in selected_zone_names_equivalences_dict.items():
    if any(acronym.strip() for acronym in value["Acronym"]):
        print(f"Key: {key};    Acronym: {value['Acronym']};    Name: {value['name']}\n")

Key: UK;    Acronym: ['GB'];    Name: ['United Kingdom']



<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    4. Data Reference Year 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Setting the variable on the target year which formatting data is wanted to
</div>
<hr style="border: 1px solid skyblue;">

In [114]:
# 1. --------------------------------------------------------------------- Set the year to which data is formating to: 
data_target_year = '2030'

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("Selected Year:", data_target_year)

Selected Year: 2030


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [113]:
print (f"Name of the DispaSET Unleash folder:                      {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                      {dispaSET_unleash_folder_path}\n")
print (f"Name of the NTC Base data folder:                         {ntc_base_data_folder_name}\n")
print (f"Path to the NTC Base data folder:                         {ntc_base_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Raw data 1 folder:                  {ntc_pypsa_raw_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Raw data 1 folder:                  {ntc_pypsa_raw_data_folder_path}\n")
print (f"Name of the NTC_Matrix data folder:                       {ntc_matrix_raw_data_folder_name}\n")
print (f"Path to the NTC_Matrix data folder:                       {ntc_matrix_raw_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Formated data folder:               {ntc_pypsa_formated_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Formated data folder:               {ntc_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                        {zone_names}\n")
print (f"Name of the zone_names_equivalences_dictionary):          {[value['Acronym'] for value in selected_zone_names_equivalences_dict.values()]}\n")
print (f"Target year:                                              {data_target_year}\n")
print (f"Target time step:                                         {data_target_time_step}\n")

Name of the DispaSET Unleash folder:                      Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                      /home/ray/Dispa-SET_Unleash

Name of the NTC Base data folder:                         1h

Path to the NTC Base data folder:                         /home/ray/Dispa-SET_Unleash/Database/NTC/1h

Name of the NTC_Pypsa Raw data 1 folder:                  NTC

Path to the NTC_Pypsa Raw data 1 folder:                  /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/NTC

Name of the NTC_Matrix data folder:                       Raw_Data_Sources

Path to the NTC_Matrix data folder:                       /home/ray/Dispa-SET_Unleash/RawData/NTC/Raw_Data_Sources

Name of the NTC_Pypsa Formated data folder:               1h

Path to the NTC_Pypsa Formated data folder:               /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/NTC/1h

Name of the zones:                                        ['BE', 'FR', 'DE', 'NL', 'UK']

Name of th

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
4. Net Transfer Capacities Data Frame
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Creating the data frame with all the corresponding headers according the Dispa-SET nomenclature.
</div>
<div style="text-align: justify; margin-left: 4.5em; font-weight: bold; font-size: 15px; font-family: TimesNewRoman; color:skyblue">
Sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
The Dispa-SET documentation.
            <br>
            <a href="https://www.dispaset.eu/en/latest/data.html#:~:text=Interconnections%EF%83%81" style="color:skyblue">https://www.dispaset.eu/en/latest/data.html#:~:text=Interconnections%EF%83%81</a>
        </li>
</div>
<hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
4.1. Empty Zone Data Frames
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Creating empty dataframes—column headers only—for the selected zones, corresponding to the target year.
<br>
Loading the csv bade file.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [82]:
# 1. ------------------------------------------------------------------------------------ Convert data year to integer 
data_target_year = int(data_target_year)  

# 2. ----------------------------------------------------------------------------- Find the most appropriate csv files
csv_files = [
    f for f in os.listdir(ntc_base_data_folder_path)
    if f.lower().endswith(".csv")
]

# 3. --------------------------------------------------------------------- Keep only filenames that are exactly a year
available_years = {
    int(os.path.splitext(f)[0]): f
    for f in csv_files
    if os.path.splitext(f)[0].isdigit()
}

if not available_years:
    raise ValueError("No CSV files named with a year found")

# 4. ------------------------------------------------------------------------------- Select exact year or closest year
if data_target_year in available_years:
    selected_year = data_target_year
else:
    selected_year = min(
        available_years.keys(),
        key=lambda y: abs(y - data_target_year)
    )

selected_csv_path = os.path.join(
    ntc_base_data_folder_path,
    available_years[selected_year]
)

# 5. ----------------------------------------------------------------------------------- Load csv file into data frame
ntc_df = pd.read_csv(selected_csv_path)

# 6. ------------------------------------------------------------------------------------------------------------ Done
print(f"CSV year used: {selected_year}")
print(ntc_df.columns.tolist())
print("✅ Processing complete. Updated DataFrame stored in 'ntc_df'.")
print(ntc_df)

CSV year used: 2024
['index', 'AT -> SI', 'AT -> CZ', 'AT -> IT', 'AT -> HU', 'AT -> DE', 'AT -> CH', 'BE -> FR', 'BE -> DE', 'BE -> LU', 'BE -> UK', 'BE -> NL', 'BG -> RO', 'BG -> EL', 'CH -> DE', 'CH -> FR', 'CH -> IT', 'CH -> AT', 'CZ -> PL', 'CZ -> SK', 'CZ -> AT', 'CZ -> DE', 'DE -> NO', 'DE -> SE', 'DE -> CZ', 'DE -> FR', 'DE -> BE', 'DE -> AT', 'DE -> DK', 'DE -> LU', 'DE -> CH', 'DE -> PL', 'DE -> NL', 'DK -> DE', 'DK -> SE', 'DK -> NO', 'DK -> UK', 'DK -> NL', 'EE -> LV', 'EE -> FI', 'EL -> IT', 'EL -> BG', 'ES -> FR', 'ES -> PT', 'FI -> SE', 'FI -> NO', 'FI -> EE', 'FR -> BE', 'FR -> IT', 'FR -> ES', 'FR -> DE', 'FR -> UK', 'FR -> CH', 'HR -> SI', 'HR -> HU', 'HU -> SI', 'HU -> RO', 'HU -> SK', 'HU -> AT', 'HU -> HR', 'IE -> UK', 'IT -> SI', 'IT -> FR', 'IT -> EL', 'IT -> AT', 'IT -> CH', 'IT -> MT', 'LT -> PL', 'LT -> SE', 'LT -> LV', 'LU -> BE', 'LU -> DE', 'LV -> EE', 'LV -> LT', 'MT -> IT', 'NL -> DE', 'NL -> NO', 'NL -> BE', 'NL -> DK', 'NL -> UK', 'NO -> DE', 'NO -> SE'

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Identifying the interconnections between the selected zones/countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [83]:
# 1. --------------------------------------------------------------------- Build all ordered zone combinations (A -> B)
zone_combinations = [f"{a} -> {b}" for a, b in permutations(zone_names, 2)]
zone_combinations_set = set(zone_combinations)

# 2. ------------------------- --------------------------------- Identify columns that look like a combination (X -> Y)
combination_like_cols = [
    col for col in ntc_df.columns
    if isinstance(col, str) and " -> " in col
]

# 3. ------------------------------------------------------ Among the combination-like columns, keep only exact matches
matching_cols = [col for col in combination_like_cols if col in zone_combinations_set]

# 4. ---------------------------------------------------------------- Remove combination-like columns that do NOT match
cols_to_drop = set(combination_like_cols) - set(matching_cols)
ntc_df = ntc_df.drop(columns=cols_to_drop)

# 5. --------------------------------------------------------------------- Create missing combinations as empty columns
missing_combinations = zone_combinations_set - set(matching_cols)
for col in missing_combinations:
    ntc_df[col] = np.nan

# 6. ---------------------------------------- Sort columns if desired: keep original order + new combinations at the end
ntc_df = ntc_df[list(ntc_df.columns)]

# 7. -------------------------------------------------------------------------------------------------------------- Done
print("✅ Processing complete. Updated DataFrame stored in 'ntc_df'.")
print(ntc_df)

                          index  BE -> FR  BE -> DE  BE -> UK  BE -> NL  \
0     2024-01-01 00:00:00+00:00    2372.0     506.5     523.0    1603.0   
1     2024-01-01 01:00:00+00:00    2372.0     506.5     523.0    1603.0   
2     2024-01-01 02:00:00+00:00    2372.0     506.5     523.0    1603.0   
3     2024-01-01 03:00:00+00:00    2372.0     506.5     523.0    1603.0   
4     2024-01-01 04:00:00+00:00    2372.0     506.5     523.0    1603.0   
...                         ...       ...       ...       ...       ...   
8779  2024-12-31 19:00:00+00:00    2372.0     506.5     523.0    1603.0   
8780  2024-12-31 20:00:00+00:00    2372.0     506.5     523.0    1603.0   
8781  2024-12-31 21:00:00+00:00    2372.0     506.5     523.0    1603.0   
8782  2024-12-31 22:00:00+00:00    2372.0     506.5     523.0    1603.0   
8783  2024-12-31 23:00:00+00:00    2372.0     506.5     523.0    1603.0   

      DE -> FR  DE -> BE  DE -> NL  FR -> BE  FR -> DE  ...  NL -> DE  \
0       1707.0     501.5  

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
4.2. Time Step Correction
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Verifying and correcting first column timestamps in the dataframe to match target year and selected time step.
<hr style="border: 1px solid skyblue;">
</div>

In [84]:
# 1. ---------------------------------------------- Define a mapping for time step strings to pandas frequency strings
time_step_map = {
    "1h"   : "H",
    "15min": "15T",
    "30min": "30T"
}

# 2. -------------------------------------------------------- Get the pandas frequency string for the target time step
freq = time_step_map[data_target_time_step]

# Assume `df` is your single DataFrame
if ntc_df.empty:
    print("DataFrame is empty. Skipping alignment.")
else:
    
# 2.1 -------------------------------------------------------------------------------------- Identify the first column
    first_col = ntc_df.columns[0]

# 2.2 ------------------------------------------------------------- Convert column to datetime with UTC if not already
    ntc_df[first_col] = pd.to_datetime(ntc_df[first_col], utc=True, errors='coerce')

# 2.3 --------------------------------------------------------------------------- Remove any rows that failed to parse
    ntc_df = ntc_df.dropna(subset=[first_col])

# 2.4 --------------------------------------------------------- Create a date range for the correct year and time step
    start_time = pd.Timestamp(f"{data_target_year}-01-01 00:00:00", tz="UTC")
    end_time = pd.Timestamp(f"{data_target_year}-12-31 23:59:59", tz="UTC")

    correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)

# 2.5 --------------------------------------------------------- Replace the first column with the corrected date range
    if len(correct_index) >= len(ntc_df):
        ntc_df[first_col] = correct_index[:len(ntc_df)]
    else:
# 2.5.1 ----------------------------------------- If df has more rows than the date range, extend with repeated values
        repeats = (len(ntc_df) // len(correct_index)) + 1
        ntc_df[first_col] = pd.Series(list(correct_index) * repeats)[:len(ntc_df)]

# 2.6 ----------------------------------------------------------------------------------------------------------- Done
    print(f"Updated DataFrame: first column aligned with {data_target_year} and {data_target_time_step}")

Updated DataFrame: first column aligned with 2030 and 1h


/tmp/ipykernel_2152992/672602477.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [126]:
print (f"Name of the DispaSET Unleash folder:                      {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                      {dispaSET_unleash_folder_path}\n")
print (f"Name of the NTC Base data folder:                         {ntc_base_data_folder_name}\n")
print (f"Path to the NTC Base data folder:                         {ntc_base_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Raw data 1 folder:                  {ntc_pypsa_raw_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Raw data 1 folder:                  {ntc_pypsa_raw_data_folder_path}\n")
print (f"Name of the NTC_Matrix data folder:                       {ntc_matrix_raw_data_folder_name}\n")
print (f"Path to the NTC_Matrix data folder:                       {ntc_matrix_raw_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Formated data folder:               {ntc_pypsa_formated_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Formated data folder:               {ntc_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                        {zone_names}\n")
print (f"Name of the zone_names_equivalences_dictionary):          {[value['Acronym'] for value in selected_zone_names_equivalences_dict.values()]}\n")
print (f"Target year:                                              {data_target_year}\n")
print (f"Target time step:                                         {data_target_time_step}\n")
print (f"Name of the NTC DataFrame:                                {list(ntc_df.keys())}\n")

Name of the DispaSET Unleash folder:                      Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                      /home/ray/Dispa-SET_Unleash

Name of the NTC Base data folder:                         1h

Path to the NTC Base data folder:                         /home/ray/Dispa-SET_Unleash/Database/NTC/1h

Name of the NTC_Pypsa Raw data 1 folder:                  NTC

Path to the NTC_Pypsa Raw data 1 folder:                  /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/NTC

Name of the NTC_Matrix data folder:                       Raw_Data_Sources

Path to the NTC_Matrix data folder:                       /home/ray/Dispa-SET_Unleash/RawData/NTC/Raw_Data_Sources

Name of the NTC_Pypsa Formated data folder:               1h

Path to the NTC_Pypsa Formated data folder:               /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/NTC/1h

Name of the zones:                                        ['BE', 'FR', 'DE', 'NL', 'UK']

Name of th

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: 'Times New Roman', serif; color: skyblue;">
5. PyPSA vs Dispaset Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
All those technologies from PyPSA which can be represented in Dispa-SET have to be identified.
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.1. PyPSA Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
The next chart represents graphically how all the Energy sector inside PyPSA is structured.<br>
This is used to get the equivalent diagram for Dispaset.
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_multisector_figure_1.png" 
       alt="PyPSA Multisector Flow Diagram" 
       style="max-width:35%; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: <a href="https://pypsa-eur.readthedocs.io/en/latest/" target="_blank" style="color: skyblue; text-decoration: underline;">PyPSA-Eur Documentation</a>
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.2. Equivalent Dispa-SET Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
A correlation has been established between the PyPSA parameters and their corresponding Dispa-SET equivalents:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
Due to feature limitations, not all elements from PyPSA can be represented in Dispa-SET.
<br>
However, for those compatible technologies, the following chart graphically illustrates how they are connected within the Dispa-SET environment logic:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_Filtered_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.3. Technologies, Demmands & Interconnection Lines Nomenclature - PyPSA vs Dispaset 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The PyPSA technologies and demmands nomenclature and their correlation with their homologous from Dispa-SET are described as follows:
</div>
<table style="width: 95%; margin-left: auto; margin-right: auto; border-collapse: collapse; font-family: TimesNewRoman; font-size: 12px; color: skyblue;">
  <thead>
    <tr style="background-color: #1E1E1E; color: skyblue; border-bottom: 1px solid skyblue;">
      <th style="width: 8%; padding: 8px; text-align: left; border: 1px solid #444;">PyPSA Element</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Technology</th>
      <th style="width: 10%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Classification</th>
      <th style="width: 34%; padding: 8px; text-align: left; border: 1px solid #444;">Description</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Relation</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Dispa-SET Element</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Element Type</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">May Modeled?</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DC</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents the DC (HVDC) transmission network for electricity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">NTC</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">OCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Open-Cycle Gas Turbine producing electricity from gas.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined-Cycle Gas Turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">EV charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Interface between the grid and electric vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">V2G</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Vehicle-to-Grid---allows EVs to discharge electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to stored energy in batteries (charging link)</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">BioSNG</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces synthetic natural gas (bio-methane)---fuel synthesis process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DAC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct Air Capture---captures CO<sub>2</sub> for storage or utilization</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Fischer-Tropsch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + CO<sub>2</sub> to liquid hydrocarbons; fuel synthesis</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Electrolysis</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity into hydrogen cross-sector conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Fuel Cell</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts hydrogen back to electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports hydrogen between regions or sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline retrofitted</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Existing pipelines adapted for H<sub>2</sub> transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Generates electricity or heat from hydrogen---boundary technology</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Haber-Bosch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + N<sub>2</sub> into ammonia---chemical/fertilizer industry</td>
      <td style="padding: 8px; border: 1px solid #444;">PX2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Steam Methane Reforming---gas to hydrogen conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR with Carbon Capture---industrial hydrogen with CC</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Sabatier</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> + CO<sub>2</sub> → CH<sub>4</sub>---synthetic methane production.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture machinery oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil used in agricultural machinery---transport/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">ammonia cracker</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts ammonia back into hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored electricity from batteries</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity distribution grid</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents distribution-level power flow---low voltage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">nuclear</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear-to-electricity conversion within the power system.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Upgrades raw biogas into pipeline-quality methane</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biogas upgrading with carbon capture---industrial fuel conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biomass to liquid</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass into liquid fuels---synthetic fuel process.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents captured Tons of CO<sub>2</sub> / hour transported or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Use of coal as industrial feedstock/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Supplies natural gas to industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial gas use with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports natural gas---energy carrier infrastructure</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline new</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Expansion of natural gas transport capacity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">kerosene for aviation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Aviation fuel consumption---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil use for land transport---transport fuel consumption</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">methanolisation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Synthesizes methanol (CO<sub>2</sub> + H<sub>2</sub> → CH<sub>3</sub>OH)</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides naphtha feedstock to industrial processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial process CO<sub>2</sub> emissions. Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial CO<sub>2</sub> emissions with capture — Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for rural homes</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass to heat---residential fuel use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---non-electric final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Uses electricity directly for heating---part of demand side</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to thermal energy in storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat — part of residential heating loop</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized air-source heat pump for urban buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating devices---boundary heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">link & residential rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heat using stable ground temperature</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
     <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges heat from thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Service sector rural building heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides space or process heat for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity-to-heat for service buildings---heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating in rural service buildings.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating for service buildings---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for service‐sector thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to buildings---part of the heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Local electric heat production for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass‐to‐heat conversion---end-use heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct electric heating for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts power to stored heat</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Releases stored thermal energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Methanol use in maritime transport---fuel demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption for ships---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass fuel use for industrial heat/processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same as above but with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass transport</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents biomass logistics between regions/sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">FlowXmaximum & FlowXminimum</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized district heat pump---heat sector interfac.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined heat + power supplying district heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Gas CHP with carbon capture---district heating system</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized gas heating for urban networks</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric boiler for district heating---end-use conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass combined heat + power---heat boundary process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same with carbon capture---boundary sector</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat in district storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to the district network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fossil fuel-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas–fired power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal variant used for power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
        <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Powerx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">onwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Onshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-ac</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with AC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-dc</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with DC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Utility-scale photovoltaic generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar rooftop</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed PV connected to power grid</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">ror</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Run-of-river hydro power plant</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel input for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces heat for households (not electricity)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized solar heating for buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Solar thermal for service-sector heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban service-sector solar heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized solar thermal for district heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">load</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents total shredding energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Load Shedding</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">hydro</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional hydro reservoir</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">PHS</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Pumped Hydro Storage, a grid-scale electricity storage technology</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">battery</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electrical energy storage — directly coupled with the grid.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
            <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel stock for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen storage---chemical energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">NH<sub>3</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Ammonia storage---chemical/fertilizer or fuel vector.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomethane stock for heating or industry</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Captured CO<sub>2</sub> pool---used in synthesis or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Permanent CO<sub>2</sub> storage---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>      
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> stored</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Intermediate or final CO<sub>2</sub> reservoir---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal stock for industrial/fuel processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas (CH<sub>4</sub>) stock — cross-sector energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fuel storage for thermal use — outside grid operations</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Liquid fuel stock — used in transport or synthesis chains</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil (synthetic hydrocarbons) stock for transport/industrial fuels</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass stock for heating/industrial use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for rural households — heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1评审44;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed heat storage in urban residences</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat storage for rural service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for urban service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating storage — boundary heat network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Final oil demand in the industrial sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary DH demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating demand for residential and service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized residential space/water heating demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in lighting, irrigation, machinery)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat energy required in agricultural processes (drying, greenhouses)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption in agricultural machinery and vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">aviation oil demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Jet fuel (kerosene) demand for aviation transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand for rail network</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Traction electricity used by rail and metro transport systems</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand of residential and tertairy</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in households and service-sector buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity use for machinery, processes, and electrified production</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">methane</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas or synthetic methane Industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">hydrogen for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand for industrial refining, ammonia, steelmaking</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport EV</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption by electric vehicles in road transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport hydrogen demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen fuel demand for road transport (fuel-cell vehicles)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">low-temperature heat for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">(below $\sim 200^{\circ}$C), typically supplied by boilers or heat pumps</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">Non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Naphtha used as a chemical feedstock e.g., plastics, petrochemicals</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">oil to transport demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil demand for conventional land gasoline and diesel vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand, maritime transport fuel-cell/ combustion ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional marine oil fuel demand (HFO, MGO) for ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass demand in industrial processes for heat or material use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
  </tbody>
</table>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 0.5px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.3. Dispa-SET vs PyPSA Interconnection Lines Equivalences Dictionary
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
To facilitate data harmonization, a dictionary containing the correlations between the raw sources files and the Dispa-SET and PyPSA nomenclature equivalences for interconnection lines is developed, leveraging the specifications detailed in the preceding table.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [86]:
# 1. ------------------------------------------------------- Dictionary mapping Dispa-SET acronyms to PyPSA tech names 
tech_equivalences_dict = {
    
                            # -----------------------
                            # Interconection lines
                            # -----------------------
                            "Tech_AC"  :  {"transmission_lines": ["AC"  ] ,   "capacities": ["AC Transmission lines"  ],   "RM_Factor": ["0.898"  ] } ,
    
                            "Tech_DC"  :  {"transmission_lines": ["DC"  ] ,   "capacities": ["DC Transmission lines"  ],   "RM_Factor": ["1.000"  ] } ,
       
                          }

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print(tech_equivalences_dict)

{'Tech_AC': {'transmission_lines': ['AC'], 'capacities': ['AC Transmission lines'], 'RM_Factor': ['0.898']}, 'Tech_DC': {'transmission_lines': ['DC'], 'capacities': ['DC Transmission lines'], 'RM_Factor': ['1.000']}}


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
        6. Interconnection Data Sources
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
        Two types of data sources are available for country interconnections:
    </div>
    <div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
        <li>
        Detailed_Source: Provides a disaggregated view by country pair lines (from emitter to receptor).
        <li>
        Aggregated_Source: Provides the total capacity of all interconnection lines by country (only emitter).
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
        The processing logic prioritizes detailed sources to ensure higher data accuracy. If only aggregated data is available, the code performs a proportional disaggregation for each country pair based on a predefined supporting matrix.
    </div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.1. Aggregate Sources Uploading
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
Begin with the less accurate sources; if more detailed files are later found, the formatted data will be overwritten with the updated information.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [117]:
# 1. ------------------------------------------------------------------------------------- Initialize output dictionary
nodal_capacities_dfs_dict = {}

# 2. ------------------------------------------------------------------------- List all CSV files in the directory once
available_csv_files = [
    f for f in os.listdir(ntc_pypsa_raw_data_folder_path)
    if f.lower().endswith(".csv")
]

# 2.1. ------------------------------------------------------------------------------------ Check if any CSV files exist
if not available_csv_files:
    print("Warning: No CSV files found in the directory.")
else:
    print(f"Info: Found {len(available_csv_files)} CSV file(s) in the directory: {available_csv_files}")

# 3. ----------------------------------------------------------------------------------------------- Iterate over zones
for zone in zone_names:
    
# 3.1. ---------------------------------------------------------------------------------------- Build candidate strings
    candidate_strings = [zone]

    if zone in zone_names_equivalences_dict:
        acronyms = zone_names_equivalences_dict[zone].get("Acronym", [])
        candidate_strings.extend([ac.strip() for ac in acronyms if ac.strip()])

# 3.2. ------------------------------------------------------------------------------ Find the first matching CSV file
    selected_file = None
    for f in available_csv_files:
        for candidate in candidate_strings:
            if candidate.lower() in f.lower():
                selected_file = f
                break
        if selected_file:
            break

# 3.3. --------------------------------------------------------------------------------------------- Load CSV if found
    if selected_file:
        csv_path = os.path.join(ntc_pypsa_raw_data_folder_path, selected_file)
        df_name = f"{zone}_{os.path.splitext(selected_file)[0].split('_', 1)[1]}" if "_" in selected_file else f"{zone}_data"
        nodal_capacities_dfs_dict[df_name] = pd.read_csv(csv_path)

# 4. ------------------------------------------------------------------------------------------------------------ Done
print("✅ Processing complete. Updated DataFrames stored in 'nodal_capacities_dfs_dict'.")
print(nodal_capacities_dfs_dict)

Info: Found 8 CSV file(s) in the directory: ['transmission_lines_suff_2050.csv', 'transmission_lines_suff_2030.csv', 'NL_capacities.csv', 'transmission_lines_suff_2040.csv', 'BE_capacities.csv', 'FR_capacities.csv', 'DE_capacities.csv', 'GB_capacities.csv']
✅ Processing complete. Updated DataFrames stored in 'nodal_capacities_dfs_dict'.
{'BE_capacities':                          tech          2019          2030          2040  \
0       AC Transmission lines  7.700000e+03  1.039500e+04  1.403324e+04   
1                      BioSNG  0.000000e+00  2.538195e-02  1.320336e+03   
2                        CCGT  4.267700e+03  6.279421e+03  3.037359e+03   
3                         CCS  0.000000e+00  6.541376e+02  9.200629e+02   
4                         CHP  2.408720e+03  5.240790e+03  5.241222e+03   
5       DC Transmission lines  2.000000e+03  2.699998e+03  3.644998e+03   
6                  EV charger  0.000000e+00  3.711562e+03  1.360906e+04   
7                          H2  0.000000e+00

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Filtering the target year.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [118]:
# 1. -------------------------------------------------------------------------------------------- Filter each DataFrame
if not nodal_capacities_dfs_dict:
    print("Warning: nodal_capacities_dfs_dict is empty. No DataFrames to filter.")
else:
    for zone, df in nodal_capacities_dfs_dict.items():
    
# 1.1. --------------------------------------------------------------------------------------- Identify numeric columns
        numeric_cols = [col for col in df.columns if str(col).isdigit()]

# 1.2. ----------------------------------------------------- Keep only the numeric column that matches data_target_year
        numeric_cols_to_keep = [col for col in numeric_cols if str(col) == str(data_target_year)]

# 1.3. ------------------------------------------------------------------------------- Combine with non-numeric columns
        cols_to_keep = [col for col in df.columns
                        if (str(col).startswith("zone") or
                            str(col).startswith("tech") or
                            col in numeric_cols_to_keep)]

# 1.4. ------------------------------------------------------------------------------------------- Filter the DataFrame
        nodal_capacities_dfs_dict[zone] = df[cols_to_keep].copy()

# 2. ------------------------------------------------------------------------------------------------------------- Done
print("✅ Processing complete. Updated DataFrames stored in 'nodal_capacities_dfs_dict'.")
print(nodal_capacities_dfs_dict)

✅ Processing complete. Updated DataFrames stored in 'nodal_capacities_dfs_dict'.
{'BE_capacities':                          tech          2030
0       AC Transmission lines  1.039500e+04
1                      BioSNG  2.538195e-02
2                        CCGT  6.279421e+03
3                         CCS  6.541376e+02
4                         CHP  5.240790e+03
5       DC Transmission lines  2.699998e+03
6                  EV charger  3.711562e+03
7                          H2  4.204135e-01
8                  H2 turbine  2.261108e-01
9                        OCGT  2.671197e+03
10                        SMR  2.639505e-02
11                        V2G  3.711562e+03
12  agriculture machinery oil  3.443462e+02
13            ammonia cracker  1.914452e-02
14                    battery  1.806048e+04
15            battery storage  7.151214e+03
16                     biogas  1.338338e+07
17                    biomass  2.094568e+07
18             biomass boiler  1.325200e+03
19          biomass t

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [127]:
print (f"Name of the DispaSET Unleash folder:                        {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                        {dispaSET_unleash_folder_path}\n")
print (f"Name of the NTC Base data folder:                           {ntc_base_data_folder_name}\n")
print (f"Path to the NTC Base data folder:                           {ntc_base_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Raw data 1 folder:                    {ntc_pypsa_raw_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Raw data 1 folder:                    {ntc_pypsa_raw_data_folder_path}\n")
print (f"Name of the NTC_Matrix data folder:                         {ntc_matrix_raw_data_folder_name}\n")
print (f"Path to the NTC_Matrix data folder:                         {ntc_matrix_raw_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Formated data folder:                 {ntc_pypsa_formated_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Formated data folder:                 {ntc_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                          {zone_names}\n")
print (f"Name of the zone_names_equivalences_dictionary):            {[value['Acronym'] for value in selected_zone_names_equivalences_dict.values()]}\n")
print (f"Target year:                                                {data_target_year}\n")
print (f"Target time step:                                           {data_target_time_step}\n")
print (f"Name of the NTC DataFrame:                                  {list(ntc_df.keys())}\n")
print (f"Name of the Technology Match Dictionary:                    tech_equivalences_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary): {list(nodal_capacities_dfs_dict.keys())}\n")

Name of the DispaSET Unleash folder:                        Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                        /home/ray/Dispa-SET_Unleash

Name of the NTC Base data folder:                           1h

Path to the NTC Base data folder:                           /home/ray/Dispa-SET_Unleash/Database/NTC/1h

Name of the NTC_Pypsa Raw data 1 folder:                    NTC

Path to the NTC_Pypsa Raw data 1 folder:                    /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/NTC

Name of the NTC_Matrix data folder:                         Raw_Data_Sources

Path to the NTC_Matrix data folder:                         /home/ray/Dispa-SET_Unleash/RawData/NTC/Raw_Data_Sources

Name of the NTC_Pypsa Formated data folder:                 1h

Path to the NTC_Pypsa Formated data folder:                 /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/NTC/1h

Name of the zones:                                          ['BE', 'FR', 'DE', '

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    6.1.1. PyPSA to Dispa-SET Country Interconnections Data Formatting
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The NTC dataframe start to be formatting.
    </div>
    <hr style="border: 1px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
    6.1.2. Interconnection Types Classification
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
All technical data points unrelated to interconnection infrastructure is removed for all countries.
    <br>
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [119]:
# 1. ----------------------------------------------------------- Extract the COMMON PART of tech_equivalences_dict keys
tech_common_parts = set(
    key.split("_")[0] for key in tech_equivalences_dict.keys()
)

# 1.1. ------------------------------------------------------------------------------- Check if the dictionary is empty
if not nodal_capacities_dfs_dict:
    print("Warning: nodal_capacities_dfs_dict is empty. No DataFrames to process.")
else:
    
# 2. -------------------------------------------------------------------------------------- Iterate over the dictionary
    for df_key, df in nodal_capacities_dfs_dict.items():

# 2.1 --------------------------------------------------------- Identify the technology column (similar to common part)
        tech_column = None
        for col in df.columns:
            for common_part in tech_common_parts:
                if common_part.lower() in str(col).lower():
                    tech_column = col
                    break
            if tech_column is not None:
                break

# 2.2. ------------------------------------------------------------------- If no tech column found, skip this DataFrame
        if tech_column is None:
            continue

# 2.3. --------------------------- Identify applicable sub-keys in dictionary (those whose sub-key name matches df_key)
        allowed_values = set()
        value_to_key_map = {}  # Map to store the replacement key for each allowed value

        for tech_key, tech_data in tech_equivalences_dict.items():
            for subkey_name, subkey_values in tech_data.items():

# 2.3.1. ------------------------------------------------------------ subkey name must be similar (not equal) to df_key
                if subkey_name.lower() in df_key.lower():
                    for val in subkey_values:
                        allowed_values.add(val)
                        value_to_key_map[val] = tech_key  # Map the value to its corresponding key

# 2.4. --------------------------------------------------------------------- If no allowed values found, skip filtering
        if not allowed_values:
            continue

# 2.5. ------------------------------------------------------------------------------------------- Filter the DataFrame
        df_filtered = df[df[tech_column].isin(allowed_values)].copy()

# 2.6. --------------------------------------------------------------------------------------- Replace values with keys
        df_filtered[tech_column] = df_filtered[tech_column].map(value_to_key_map)

# 2.7. --------------------------------------------------------------------------------- Replace the original DataFrame
        nodal_capacities_dfs_dict[df_key] = df_filtered.reset_index(drop=True)
        print(f"Info: Successfully processed DataFrame '{df_key}'.")

# 3. ------------------------------------------------------------------------------------------------------------- Done
print("✅ Processing complete. Updated DataFrames stored in 'nodal_capacities_dfs_dict'.")
print(nodal_capacities_dfs_dict)

Info: Successfully processed DataFrame 'BE_capacities'.
Info: Successfully processed DataFrame 'FR_capacities'.
Info: Successfully processed DataFrame 'DE_capacities'.
Info: Successfully processed DataFrame 'NL_capacities'.
Info: Successfully processed DataFrame 'UK_capacities'.
✅ Processing complete. Updated DataFrames stored in 'nodal_capacities_dfs_dict'.
{'BE_capacities':       tech          2030
0  Tech_AC  10394.997796
1  Tech_DC   2699.998415, 'FR_capacities':       tech          2030
0  Tech_AC  10259.992768
1  Tech_DC   9479.722548, 'DE_capacities':       tech          2030
0  Tech_AC  10782.930620
1  Tech_DC   3239.996962, 'NL_capacities':       tech          2030
0  Tech_AC  10917.935647
1  Tech_DC   2969.997804, 'UK_capacities':       tech          2030
0  Tech_AC      0.000000
1  Tech_DC  19403.563179}


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
        6.1.3. Reliability Margin Factor Influence
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
        Reliability Margin (RM) is the overarching term used to describe a capacity margin that ensures secure grid operation.
        <br>
        It acts as a “margin of error” to cover discrepancies between forecasted electricity flows and real-time flows, accounting for unforeseen external influences.
        <br>
        It is a critical component used to calculate the Net Transfer Capacity (NTC).
    </div>
    <div style="text-align: center; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue; margin-top: 10px; margin-bottom: 10px;">
        $NTC = TTC - RM$
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
        Where $NTC$ is the Net Transfer Capacity, $TTC$ is the Total Transfer Capacity, and $RM$ is the Reliability Margin.
<br>
        The following percentual values are used for each interconnection type:
    </div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
RM  ≈  10.2 % of the TTC; for AC
</li>
<li>
RM  ≈  00.0 % of the TTC; for DC
</li>
</div>
        <div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
            <ol style="margin-top: 10px; padding-left: 1.5em;">
                <li>
                    ENTSO-E: NTC Users Information.
                    <br>
                    <a href="https://eepublicdownloads.entsoe.eu/clean-documents/pre2015/ntc/entsoe_NTCusersInformation.pdf" style="color:skyblue">https://eepublicdownloads.entsoe.eu/clean-documents/pre2015/ntc/entsoe_NTCusersInformation.pdf</a>
                </li>
                <br>
                <li>
                    ENTSO-E: Capacity Calculation & Allocation (CC&A) Report 2023.
                    <br>
                    <a href="https://eepublicdownloads.blob.core.windows.net/public-cdn-container/clean-documents/nc-tasks/ENTSO-E%20CC&A%20Report%202023.pdf" style="color:skyblue">https://eepublicdownloads.blob.core.windows.net/public-cdn-container/clean-documents/nc-tasks/ENTSO-E%20CC&A%20Report%202023.pdf</a>
                </li>
                <br>
                <li>
                    SEE CCR TSOs: Proposal for Capacity Calculation Methodology - Explanatory Note.
                    <br>
                    <a href="https://consultations.entsoe.eu/markets/see-ccr-tsos-proposal-of-ccm/supporting_documents/SEE%20CCR%20Explanatory%20Note%20_public%20consultation.pdf" style="color:skyblue">https://consultations.entsoe.eu/markets/see-ccr-tsos-proposal-of-ccm/supporting_documents/SEE%20CCR%20Explanatory%20Note%20_public%20consultation.pdf</a>
                </li>
            </ol>
        </div>
    <hr style="border: 0.5px solid skyblue;">
</div>

In [120]:
# 1. ------------------------------------------------------------------------------ extract common parts from tech keys
common_parts = set()
for key in tech_equivalences_dict.keys():
    common_part = key.split('_')[0].lower()  # lowercase for case-insensitive matching
    common_parts.add(common_part)

# 1.1. ------------------------------------------------------------------------------- Check if the dictionary is empty
if not nodal_capacities_dfs_dict:
    print("Warning: nodal_capacities_dfs_dict is empty. No DataFrames to process.")
else:
    
# 2. ------------------------------------------------------------------------------------------ loop through dataframes
    for zone_df_key, df in nodal_capacities_dfs_dict.items():
        col_to_update = None
        for col in df.columns:
            
# 2.1. -------------------------------------- check if any common part is a substring of the column name OR vice versa
            for common_part in common_parts:
                if common_part in col.lower() or col.lower() in common_part:
                    col_to_update = col
                    break
            if col_to_update:
                break
        
        if not col_to_update:
            continue  # skip if no matching column

# 2.2. ------------------------------------------------------------------------------------------ update the dataframe
        for tech_key, sub_dict in tech_equivalences_dict.items():
            
# 2.2.1. ---------------------------------------------------------------------------------------- detect numeric value
            numeric_value = None
            for v in sub_dict.values():
                if isinstance(v, list) and len(v) == 1:
                    item = v[0]
                    if isinstance(item, (int, float)):
                        numeric_value = item
                        break
                    if isinstance(item, str) and item.replace('.', '', 1).isdigit():
                        numeric_value = float(item)
                        break
                elif isinstance(v, (int, float)):
                    numeric_value = v
                    break
            
            if numeric_value is None:
                raise ValueError(f"No numeric subkey found for tech key '{tech_key}'")

# 2.2.2. ------------------------------------------------------------------------ apply multiplication and replacement
            match_mask = df[col_to_update] == tech_key
            if match_mask.any():
                # Ensure we are working on the target year column
                df.loc[match_mask, str(data_target_year)] *= numeric_value
                df.loc[match_mask, col_to_update] = 'NTC'

        print(f"Info: Successfully processed DataFrame '{zone_df_key}'.")

# 3. ------------------------------------------------------------------------------------------------------------ Done
print("✅ Processing complete. Updated DataFrames stored in 'nodal_capacities_dfs_dict'.")
print(nodal_capacities_dfs_dict)

Info: Successfully processed DataFrame 'BE_capacities'.
Info: Successfully processed DataFrame 'FR_capacities'.
Info: Successfully processed DataFrame 'DE_capacities'.
Info: Successfully processed DataFrame 'NL_capacities'.
Info: Successfully processed DataFrame 'UK_capacities'.
✅ Processing complete. Updated DataFrames stored in 'nodal_capacities_dfs_dict'.
{'BE_capacities':   tech         2030
0  NTC  9334.708020
1  NTC  2699.998415, 'FR_capacities':   tech         2030
0  NTC  9213.473506
1  NTC  9479.722548, 'DE_capacities':   tech         2030
0  NTC  9683.071696
1  NTC  3239.996962, 'NL_capacities':   tech         2030
0  NTC  9804.306211
1  NTC  2969.997804, 'UK_capacities':   tech          2030
0  NTC      0.000000
1  NTC  19403.563179}


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Total aggregated Net Transfer Capacity by country.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [122]:
# 1. ------------------------------------------------------------------------------------- Convert data year to string 
data_target_year = str(data_target_year) 

# 2. -------------------------------------------------------- Extract the common part from tech_equivalences_dict keys
common_parts = set()
for key in tech_equivalences_dict.keys():
    
# 2.1. -------------------------------------------- Split the key by '_' and take the first part, convert to lowercase
    common_part = key.split('_')[0].lower()
    common_parts.add(common_part)
common_part = common_parts.pop() if common_parts else None

# 2.2. ------------------------------------------------------------------------------ Check if the dictionary is empty
if not nodal_capacities_dfs_dict:
    print("Warning: nodal_capacities_dfs_dict is empty. No DataFrames to process.")
else:

# 3. ------------------------------------------------------------- Process each DataFrame in nodal_capacities_dfs_dict
    for zone in zone_names:
        for df_key in nodal_capacities_dfs_dict:
            if df_key.startswith(zone):
                df = nodal_capacities_dfs_dict[df_key]
            
# 3.1. -------------------------------------------------------------------- Find the column similar to the common part
                similar_column = [col for col in df.columns if common_part in col]
                if not similar_column:
                    continue
                similar_column = similar_column[0]
            
# 3.2. ----------------------------------------------- Group by the similar column and sum the data_target_year column
                grouped = df.groupby(similar_column)[data_target_year].sum().reset_index()
            
# 3.3. ------------------------------------------------------------------------------------------ Update the DataFrame
                nodal_capacities_dfs_dict[df_key] = grouped
                print(f"Info: Successfully processed DataFrame '{df_key}'.")
                
# 4. ------------------------------------------------------------------------------------------------------------ Done
if nodal_capacities_dfs_dict:
    for key, df in nodal_capacities_dfs_dict.items():
        print(f"DataFrame: {key}")
        print(df)
else:
    print("Warning: No DataFrames to display. The dictionary is empty.")

Info: Successfully processed DataFrame 'BE_capacities'.
Info: Successfully processed DataFrame 'FR_capacities'.
Info: Successfully processed DataFrame 'DE_capacities'.
Info: Successfully processed DataFrame 'NL_capacities'.
Info: Successfully processed DataFrame 'UK_capacities'.
DataFrame: BE_capacities
  tech          2030
0  NTC  12034.706436
DataFrame: FR_capacities
  tech          2030
0  NTC  18693.196054
DataFrame: DE_capacities
  tech          2030
0  NTC  12923.068658
DataFrame: NL_capacities
  tech          2030
0  NTC  12774.304015
DataFrame: UK_capacities
  tech          2030
0  NTC  19403.563179


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
6.1.4. NTC Country Deagregation
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
As this source provides aggregated NTC data at the country level, it must be disaggregated into specific interconnections.
<br>
This is achieved by applying a proportional factor for each neighboring country based on historical data from a reference year.
    </div>
    <hr style="border: 1px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
6.1.5. NTC Base Matrix
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Loading the NTC matrix with comprehensive interconnection data for European countries, utilizing the most recent historical year available.
    <br>
    </div>
    <hr style="border: 0.5px solid skyblue;">
</div>

In [123]:
# 1. ------------------------------------------------------------------------------------- Convert data year to string 
data_target_year = int(data_target_year) 

# 2. -------------------------------------------------------------------- find the folder whose name (year) is closest
available_years = []

for entry in os.listdir(ntc_matrix_raw_data_folder_path):
    entry_path = os.path.join(ntc_matrix_raw_data_folder_path, entry)
    if os.path.isdir(entry_path) and entry.isdigit():
        available_years.append(int(entry))

if not available_years:
    raise ValueError("No year-named folders found")

closest_year = min(available_years, key=lambda y: abs(y - data_target_year))

chosen_year_folder_path = os.path.join(
    ntc_matrix_raw_data_folder_path,
    str(closest_year)
)

# 3. ------------------------------------------------------------------------- find the CSV containing the target name
csv_file_path = None

for root, _, files in os.walk(chosen_year_folder_path):
    for file in files:
        if file.endswith(".csv") and ntc_pypsa_raw_data_folder_name in file:
            csv_file_path = os.path.join(root, file)
            break
    if csv_file_path:
        break

if csv_file_path is None:
    raise FileNotFoundError(
        f"No CSV file containing '{ntc_pypsa_raw_data_folder_name}' found"
    )

# 4. ----------------------------------------------------------------------------------------- load CSV into DataFrame
ntc_matrix_raw_data = pd.read_csv(csv_file_path)

# 5. ------------------------------------------------------------------------------------------------------------ Done
print("✅ Processing complete. Updated DataFrame stored in 'ntc_matrix_raw_data'.")
print(ntc_matrix_raw_data)

✅ Processing complete. Updated DataFrame stored in 'ntc_matrix_raw_data'.
   From / To       AL           AM    AT           AZ           BY    BE   BA  \
0         AL        -            0     0            0            0     0    0   
1         AM        0            -     0  0.000000001            0     0    0   
2         AT        0            0     -            0            0     0    0   
3         AZ        0  0.000000001     0            -            0     0    0   
4         BY        0            0     0            0            -     0    0   
5         BE        0            0     0            0            0     -    0   
6         BA        0            0     0            0            0     0    -   
7         BG        0            0     0            0            0     0    0   
8         HR        0            0     0            0            0     0  800   
9         CY        0            0     0            0            0     0    0   
10        CZ        0            0 

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [128]:
print (f"Name of the DispaSET Unleash folder:                        {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                        {dispaSET_unleash_folder_path}\n")
print (f"Name of the NTC Base data folder:                           {ntc_base_data_folder_name}\n")
print (f"Path to the NTC Base data folder:                           {ntc_base_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Raw data 1 folder:                    {ntc_pypsa_raw_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Raw data 1 folder:                    {ntc_pypsa_raw_data_folder_path}\n")
print (f"Name of the NTC_Matrix data folder:                         {ntc_matrix_raw_data_folder_name}\n")
print (f"Path to the NTC_Matrix data folder:                         {ntc_matrix_raw_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Formated data folder:                 {ntc_pypsa_formated_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Formated data folder:                 {ntc_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                          {zone_names}\n")
print (f"Name of the zone_names_equivalences_dictionary):            {[value['Acronym'] for value in selected_zone_names_equivalences_dict.values()]}\n")
print (f"Target year:                                                {data_target_year}\n")
print (f"Target time step:                                           {data_target_time_step}\n")
print (f"Name of the NTC DataFrame:                                  {list(ntc_df.keys())}\n")
print (f"Name of the Technology Match Dictionary:                    tech_equivalences_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary): {list(nodal_capacities_dfs_dict.keys())}\n")
print (f"Name of the NTC Matrix DataFrame:                           {list(ntc_matrix_raw_data.keys())}\n")

Name of the DispaSET Unleash folder:                        Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                        /home/ray/Dispa-SET_Unleash

Name of the NTC Base data folder:                           1h

Path to the NTC Base data folder:                           /home/ray/Dispa-SET_Unleash/Database/NTC/1h

Name of the NTC_Pypsa Raw data 1 folder:                    NTC

Path to the NTC_Pypsa Raw data 1 folder:                    /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/NTC

Name of the NTC_Matrix data folder:                         Raw_Data_Sources

Path to the NTC_Matrix data folder:                         /home/ray/Dispa-SET_Unleash/RawData/NTC/Raw_Data_Sources

Name of the NTC_Pypsa Formated data folder:                 1h

Path to the NTC_Pypsa Formated data folder:                 /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/NTC/1h

Name of the zones:                                          ['BE', 'FR', 'DE', '

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Extracting the relevant interconnection reference capacities for all selected countries from the NTC matrix.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [124]:
# 1. ------------------------------------------- Iterate over each column in ntc_df that matches the pattern 'XX -> YY'
for col in ntc_df.columns:
    if ' -> ' in col:
# 1.1. --------------------------------------------------------------------- Extract 'XX' and 'YY' from the column name
        XX, YY = col.split(' -> ')

# 1.2. -----------------------------------------------  Find the row in ntc_matrix_raw_data where 'From / To' equals XX
        row = ntc_matrix_raw_data[ntc_matrix_raw_data['From / To'] == XX]

# 1.3. --------------------------------------------------------- Find the column in ntc_matrix_raw_data that matches YY
        if YY in ntc_matrix_raw_data.columns:
            
# 1.3.1. ------------------------------------------------------ Get the value at the intersection of the row and column
            value = row[YY].values

# 1.3.2. --------------------------------------------------------------  Overwrite the column in ntc_df with this value
            if len(value) > 0:
                ntc_df[col] = value[0]

# 2. ------------------------------------------------------------------------------------------------------------- Done
print("✅ Processing complete. Updated DataFrame stored in 'ntc_df'.")
print(ntc_df)

✅ Processing complete. Updated DataFrame stored in 'ntc_df'.
                         index BE -> FR BE -> DE BE -> UK BE -> NL DE -> FR  \
0    2030-01-01 00:00:00+00:00     2200     1000     1012     2400     2500   
1    2030-01-01 01:00:00+00:00     2200     1000     1012     2400     2500   
2    2030-01-01 02:00:00+00:00     2200     1000     1012     2400     2500   
3    2030-01-01 03:00:00+00:00     2200     1000     1012     2400     2500   
4    2030-01-01 04:00:00+00:00     2200     1000     1012     2400     2500   
...                        ...      ...      ...      ...      ...      ...   
8779 2030-01-01 19:00:00+00:00     2200     1000     1012     2400     2500   
8780 2030-01-01 20:00:00+00:00     2200     1000     1012     2400     2500   
8781 2030-01-01 21:00:00+00:00     2200     1000     1012     2400     2500   
8782 2030-01-01 22:00:00+00:00     2200     1000     1012     2400     2500   
8783 2030-01-01 23:00:00+00:00     2200     1000     1012     2400    

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
6.1.6. Disaggregated NTC Values
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Proportionally disaggregating national NTC totals using historical data as a reference.
        <br>
    </div>
    <hr style="border: 0.5px solid skyblue;">
</div>

In [125]:
# 1. ---------------------------------- Ensure each 'XX -> YY' column in ntc_df has a unique value (the maximum if not)
for col in ntc_df.columns:
    if ' -> ' in col:

# 1.1. ------------------------------------------------------------- Convert column to numeric, coercing errors to NaN
        ntc_df[col] = pd.to_numeric(ntc_df[col], errors='coerce')
        unique_values = ntc_df[col].dropna().unique()
        if len(unique_values) > 1:
            max_value = max(unique_values)
            ntc_df[col] = max_value

# 1.2. ------------------------------------------------------------------------------ Check if the dictionary is empty
if not nodal_capacities_dfs_dict:
    print("Warning: nodal_capacities_dfs_dict is empty. No DataFrames to process.")
else:

# 2. ------------------------------------------------------------- Sum the unique values for each correlated dataframe
    sums = {}
    for zone in zone_names:
    
# 2.1. ------------------------------------- Find the dataframe in nodal_capacities_dfs_dict that starts with the zone
        df_key = [k for k in nodal_capacities_dfs_dict.keys() if k.startswith(zone + '_')]
        if df_key:
            df_key = df_key[0]  # Assuming only one match per zone
            correlated_cols = [c for c in ntc_df.columns if c.split(' -> ')[0] == zone]
            sum_values = sum(ntc_df[col].iloc[0] for col in correlated_cols if not pd.isna(ntc_df[col].iloc[0]))
            sums[df_key] = sum_values
            print(f"Info: Summed values for zone '{zone}'.")
        else:
            print(f"Warning: No DataFrame found for zone '{zone}'. Skipping.")

# 3. --------------------------------------------------------------- Calculate the proportional factor for each column
    for col in ntc_df.columns:
        if ' -> ' in col:
            XX, YY = col.split(' -> ')
            df_key = [k for k in nodal_capacities_dfs_dict.keys() if k.startswith(XX + '_')]
            if df_key:
                df_key = df_key[0]
                sum_values = sums[df_key]
                unique_value = ntc_df[col].iloc[0]
                factor = unique_value / sum_values
                ntc_df[col] = factor
                print(f"Info: Calculated proportional factor for column '{col}'.")
            else:
                print(f"Warning: No DataFrame found for zone '{XX}'. Skipping column '{col}'.")

# 4. -------------------------------- Multiply each column by the corresponding value from the data_target_year column
    for col in ntc_df.columns:
        if ' -> ' in col:
            XX, YY = col.split(' -> ')
            df_key = [k for k in nodal_capacities_dfs_dict.keys() if k.startswith(XX + '_')]
            if df_key:
                df_key = df_key[0]
                df = nodal_capacities_dfs_dict[df_key]
                target_year_col = str(data_target_year)
                if target_year_col in df.columns:
                    target_value = df[target_year_col].iloc[0]
                    ntc_df[col] = ntc_df[col] * target_value
                    print(f"Info: Multiplied column '{col}' by target year value.")
                else:
                    print(f"Warning: Target year column '{target_year_col}' not found in DataFrame '{df_key}'. Skipping column '{col}'.")
            else:
                print(f"Warning: No DataFrame found for zone '{XX}'. Skipping column '{col}'.")

# 5. ------------------------------------------------------------------------------------------------------------ Done
print("✅ Processing complete. Updated DataFrame stored in 'ntc_df'.")
print("Final ntc_df:")
print(ntc_df)

Info: Summed values for zone 'BE'.
Info: Summed values for zone 'FR'.
Info: Summed values for zone 'DE'.
Info: Summed values for zone 'NL'.
Info: Summed values for zone 'UK'.
Info: Calculated proportional factor for column 'BE -> FR'.
Info: Calculated proportional factor for column 'BE -> DE'.
Info: Calculated proportional factor for column 'BE -> UK'.
Info: Calculated proportional factor for column 'BE -> NL'.
Info: Calculated proportional factor for column 'DE -> FR'.
Info: Calculated proportional factor for column 'DE -> BE'.
Info: Calculated proportional factor for column 'DE -> NL'.
Info: Calculated proportional factor for column 'FR -> BE'.
Info: Calculated proportional factor for column 'FR -> DE'.
Info: Calculated proportional factor for column 'FR -> UK'.
Info: Calculated proportional factor for column 'NL -> DE'.
Info: Calculated proportional factor for column 'NL -> BE'.
Info: Calculated proportional factor for column 'NL -> UK'.
Info: Calculated proportional factor for colu

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
6.1.7. Formated NTC Data Files
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Storing the formatted and disaggregated NTC data into the designated directory within the Dispa-SET folder structure.
    <hr style="border: 0.5px solid skyblue;">
</div>

In [97]:
# 1. -------------------------------------------------------------------------------------- Ensure the directory exists
os.makedirs(ntc_pypsa_formated_data_folder_path, exist_ok=True)

# 2. ----------------------------------------------------------------------------------------- Define the CSV file path
csv_filename = f"{data_target_year}.csv"
csv_filepath = os.path.join(ntc_pypsa_formated_data_folder_path, csv_filename)

# 3. ---------------------------------------------------------------------------------------- Save the DataFrame to CSV
ntc_df.to_csv(csv_filepath, index=False)

# 4. ------------------------------------------------------------------------------------------------------------- Done
print(f"DataFrame successfully saved to: {csv_filepath}")

DataFrame successfully saved to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/NTC/1h/2030.csv


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2. Disaggregate Sources Uploading
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
If detailed country-pair interconnection data is identified, it is automatically uploaded to the formatting process.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [98]:
# 1. ------------------------------------------------------------ Find the CSV file with `data_target_year` in its name
csv_files = [f for f in os.listdir(ntc_pypsa_raw_data_folder_path) if f.endswith('.csv') and str(data_target_year) in f]
if not csv_files:
    raise FileNotFoundError(f"No CSV file found with {data_target_year} in its name in {ntc_pypsa_raw_data_folder_path}")

target_csv = csv_files[0]  # Assuming only one match; adjust if needed
csv_path = os.path.join(ntc_pypsa_raw_data_folder_path, target_csv)

# 2. ------------------------------------------------------------------------------------ Load the CSV into a DataFrame
transmission_lines_df = pd.read_csv(csv_path)

# 3. ------------------------------------------------------------------------------------------------------------- Done
print(transmission_lines_df)

   Unnamed: 0   bus0   bus1  Optimized capacity type
0           1  BE1 0  FR1 0         5804.998403   AC
1           2  BE1 0  NL1 0         4589.999392   AC
2           3  DE1 0  FR1 0         4454.994365   AC
3           4  DE1 0  NL1 0         6327.936255   AC
4       14801  DE1 0  BE1 0         1349.999326   DC
5       14814  GB0 0  NL1 0         1349.998910   DC
6       14826  GB0 0  FR1 0         2699.950352   DC
7        5581  GB0 0  GB2 0         3376.348573   DC
8        5580  GB0 0  GB2 0          337.497528   DC
9          T2  FR1 0  GB0 0         1000.007702   DC
10         T6  GB0 0  BE1 0         1349.999090   DC
11        T11  GB0 0  FR1 0         1889.864176   DC
12        T18  GB0 0  FR1 0         2000.003025   DC
13        T20  GB0 0  FR1 0         1889.897293   DC
14        T21  GB0 0  DE1 0         1889.997636   DC
15        T32  GB0 0  NL1 0         1619.998895   DC


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [129]:
print (f"Name of the DispaSET Unleash folder:                        {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                        {dispaSET_unleash_folder_path}\n")
print (f"Name of the NTC Base data folder:                           {ntc_base_data_folder_name}\n")
print (f"Path to the NTC Base data folder:                           {ntc_base_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Raw data 1 folder:                    {ntc_pypsa_raw_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Raw data 1 folder:                    {ntc_pypsa_raw_data_folder_path}\n")
print (f"Name of the NTC_Matrix data folder:                         {ntc_matrix_raw_data_folder_name}\n")
print (f"Path to the NTC_Matrix data folder:                         {ntc_matrix_raw_data_folder_path}\n")
print (f"Name of the NTC_Pypsa Formated data folder:                 {ntc_pypsa_formated_data_folder_name}\n")
print (f"Path to the NTC_Pypsa Formated data folder:                 {ntc_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                          {zone_names}\n")
print (f"Name of the zone_names_equivalences_dictionary):            {[value['Acronym'] for value in selected_zone_names_equivalences_dict.values()]}\n")
print (f"Target year:                                                {data_target_year}\n")
print (f"Target time step:                                           {data_target_time_step}\n")
print (f"Name of the NTC DataFrame:                                  {list(ntc_df.keys())}\n")
print (f"Name of the Technology Match Dictionary:                    tech_equivalences_dict\n")
print (f"Name of the PyPSA Nodal Capacities DataFrames (dictionary): {list(nodal_capacities_dfs_dict.keys())}\n")
print (f"Name of the NTC Matrix DataFrame:                           {list(ntc_matrix_raw_data.keys())}\n")
print (f"Name of the disagregated transmission lines DataFrame:      transmission_lines_df{list(transmission_lines_df.keys())}\n")

Name of the DispaSET Unleash folder:                        Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                        /home/ray/Dispa-SET_Unleash

Name of the NTC Base data folder:                           1h

Path to the NTC Base data folder:                           /home/ray/Dispa-SET_Unleash/Database/NTC/1h

Name of the NTC_Pypsa Raw data 1 folder:                    NTC

Path to the NTC_Pypsa Raw data 1 folder:                    /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/NTC

Name of the NTC_Matrix data folder:                         Raw_Data_Sources

Path to the NTC_Matrix data folder:                         /home/ray/Dispa-SET_Unleash/RawData/NTC/Raw_Data_Sources

Name of the NTC_Pypsa Formated data folder:                 1h

Path to the NTC_Pypsa Formated data folder:                 /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/NTC/1h

Name of the zones:                                          ['BE', 'FR', 'DE', '

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
        6.2.1. Reliability Margin Factor Influence
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Applying the same RM Factor values as in Section 6.1.3.
        <hr style="border: 0.5px solid skyblue;">
</div>

In [100]:
# 1. -------------------------------------------------------------------------------------- Identify the numeric column
numeric_col = None

for col in transmission_lines_df.columns:
# 1.1. ------------------------------------------------------------------------ Try converting entire column to numeric
    converted = pd.to_numeric(transmission_lines_df[col], errors="coerce")
    if converted.notna().all():
        numeric_col = col
        transmission_lines_df[col] = converted  # ensure numeric dtype
        break

if numeric_col is None:
    raise ValueError("No fully numeric column found in transmission_lines_df")

# 2. --------------------------------------------------------------------- Automatically identify the technology subkey
tech_subkey = None
tech_values = set()

# 2.1. ---------------------------------------------- Look at structure using first entry (all entries are homogeneous)
sample_subdict = next(iter(tech_equivalences_dict.values()))

for subkey, value in sample_subdict.items():
    if isinstance(value, list):

# 2.1.1. ----------------------------------------------------- Collect all values for this subkey across the dictionary
        candidate_values = set()
        for subdict in tech_equivalences_dict.values():
            candidate_values.update(subdict[subkey])

# 2.1.2. --------------------------------------------------------- Check if these values appear in any dataframe column
        for col in transmission_lines_df.columns:
            if transmission_lines_df[col].isin(candidate_values).any():
                tech_subkey = subkey
                tech_values = candidate_values
                break

    if tech_subkey is not None:
        break

if tech_subkey is None:
    raise ValueError("Could not identify the technology subkey automatically")

# 3. ------------------------------------------------------------------ Identify the technology column in the dataframe
tech_col = None

for col in transmission_lines_df.columns:
    if transmission_lines_df[col].isin(tech_values).any():
        tech_col = col
        break

if tech_col is None:
    raise ValueError("Could not identify the technology column in transmission_lines_df")

# 4. ------------------------------------------------------------------- Identify the numeric factor subkey (automatic)
factor_subkey = None

for subkey, value in sample_subdict.items():
    if isinstance(value, list) and len(value) == 1:
        item = value[0]
        if isinstance(item, (int, float)):
            factor_subkey = subkey
            break
        if isinstance(item, str) and item.replace(".", "", 1).isdigit():
            factor_subkey = subkey
            break

if factor_subkey is None:
    raise ValueError("Could not identify numeric factor subkey")

# 5. ----------------------------------------------------------------------------------------- Build the multiplier map
factor_map = {}

for subdict in tech_equivalences_dict.values():
    factor = float(subdict[factor_subkey][0])
    for tech in subdict[tech_subkey]:
        factor_map[tech] = factor

# 6. ------------------------------------------------------------------------------------------ Apply RM multiplication
mask = transmission_lines_df[tech_col].isin(factor_map)

transmission_lines_df.loc[mask, numeric_col] = (
    transmission_lines_df.loc[mask, numeric_col]
    * transmission_lines_df.loc[mask, tech_col].map(factor_map)
)

# 7. -------------------------------------------------------------------------------------- Overwrite labels with "NTC"
transmission_lines_df.loc[mask, tech_col] = "NTC"

# 8. ------------------------------------------------------------------------------------------------------------- Done
print(transmission_lines_df)

   Unnamed: 0   bus0   bus1  Optimized capacity type
0           1  BE1 0  FR1 0         5212.888566  NTC
1           2  BE1 0  NL1 0         4121.819454  NTC
2           3  DE1 0  FR1 0         4000.584940  NTC
3           4  DE1 0  NL1 0         5682.486757  NTC
4       14801  DE1 0  BE1 0         1349.999326  NTC
5       14814  GB0 0  NL1 0         1349.998910  NTC
6       14826  GB0 0  FR1 0         2699.950352  NTC
7        5581  GB0 0  GB2 0         3376.348573  NTC
8        5580  GB0 0  GB2 0          337.497528  NTC
9          T2  FR1 0  GB0 0         1000.007702  NTC
10         T6  GB0 0  BE1 0         1349.999090  NTC
11        T11  GB0 0  FR1 0         1889.864176  NTC
12        T18  GB0 0  FR1 0         2000.003025  NTC
13        T20  GB0 0  FR1 0         1889.897293  NTC
14        T21  GB0 0  DE1 0         1889.997636  NTC
15        T32  GB0 0  NL1 0         1619.998895  NTC


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
        6.2.2. Data Cleaning
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Erasing all unnecessary columns and formatting the country / zone names.
<hr style="border: 0.5px solid skyblue;">
</div>

In [101]:
# 1. ----------------------------------------------------------------------------------------  Identify columns to keep
columns_to_keep = []

# 2. ---------------------------------------------------------------------------------------- Check for numeric columns
numeric_cols = transmission_lines_df.select_dtypes(include=['float64', 'int64']).columns
columns_to_keep.extend(numeric_cols)

# 3. ------------------ Check for columns containing values with zone_names or their equivalents anywhere in the string
for col in transmission_lines_df.columns:
    if col not in columns_to_keep:  # Skip if already added as numeric
        col_values = transmission_lines_df[col].dropna().unique()
        for zone in zone_names:
            
# 3.1. -------------------------------------------------------- Check if any value contains the zone or its equivalents
            equivalents = [zone] + zone_names_equivalences_dict[zone]['Acronym'] + zone_names_equivalences_dict[zone]['name']
            if any(any(str(val).__contains__(equiv) for equiv in equivalents) for val in col_values):
                columns_to_keep.append(col)
                break

# 4. ------------------------------------------------------------------------------------- Drop the rest of the columns
transmission_lines_df = transmission_lines_df[columns_to_keep]

# 5. ----------------------------------------------------------------------------- Replace values in identified columns
for col in transmission_lines_df.columns:
    if col not in numeric_cols:  # Skip numeric columns
        for zone in zone_names:
            equivalents = [zone] + zone_names_equivalences_dict[zone]['Acronym'] + zone_names_equivalences_dict[zone]['name']
            for equiv in equivalents:
                transmission_lines_df.loc[:, col] = transmission_lines_df[col].apply(
                    lambda x: zone if isinstance(x, str) and equiv in x else x
                )

# 6. ------------------------------------------------------------------------------------------------------------- Done
print(transmission_lines_df)

    Optimized capacity bus0 bus1
0          5212.888566   BE   FR
1          4121.819454   BE   NL
2          4000.584940   DE   FR
3          5682.486757   DE   NL
4          1349.999326   DE   BE
5          1349.998910   UK   NL
6          2699.950352   UK   FR
7          3376.348573   UK   UK
8           337.497528   UK   UK
9          1000.007702   FR   UK
10         1349.999090   UK   BE
11         1889.864176   UK   FR
12         2000.003025   UK   FR
13         1889.897293   UK   FR
14         1889.997636   UK   DE
15         1619.998895   UK   NL


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
        6.2.3. Interconnected Zone Names Armonization
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Applying the DispaSET interconnection format; i.e. AA -> BB.
        <hr style="border: 0.5px solid skyblue;">
</div>

In [102]:
# 1. --------------------------------------------------------------- Identify columns containing values from zone names
selected_columns = []
for col in transmission_lines_df.columns:
    if any(transmission_lines_df[col].astype(str).str.contains(zone).any() for zone in zone_names):
        selected_columns.append(col)

# 2. ------------------------------------------------------ Join the first two selected columns into a new column 'NTC'
if len(selected_columns) >= 2:
    transmission_lines_df.loc[:, 'From -> to'] = (
        transmission_lines_df[selected_columns[0]].astype(str) + ' -> ' + transmission_lines_df[selected_columns[1]].astype(str)
    )

# 3. ------------------------------------------------------------------------------------------ Drop the source columns
transmission_lines_df = transmission_lines_df.drop(columns=selected_columns)

# 4. ------------------------------------------------------------------------------------------------------------- Done
print(transmission_lines_df)

    Optimized capacity From -> to
0          5212.888566   BE -> FR
1          4121.819454   BE -> NL
2          4000.584940   DE -> FR
3          5682.486757   DE -> NL
4          1349.999326   DE -> BE
5          1349.998910   UK -> NL
6          2699.950352   UK -> FR
7          3376.348573   UK -> UK
8           337.497528   UK -> UK
9          1000.007702   FR -> UK
10         1349.999090   UK -> BE
11         1889.864176   UK -> FR
12         2000.003025   UK -> FR
13         1889.897293   UK -> FR
14         1889.997636   UK -> DE
15         1619.998895   UK -> NL


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
        6.2.4. Total NTC Values
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Aggregating total Net Transfer Capacities for all country pairs.
<hr style="border: 0.5px solid skyblue;">
</div>

In [103]:
# 1. -------------------------------------------------------------------------------- Identify numeric and zone columns
numeric_cols = transmission_lines_df.select_dtypes(include=['float64', 'int64']).columns.tolist()

zone_cols = []
for col in transmission_lines_df.columns:
    if transmission_lines_df[col].apply(lambda x: any(zone in str(x) for zone in zone_names)).any():
        zone_cols.append(col)

# 2. ---------------------------------------------------- Ensure we have exactly one zone column and one numeric column
if len(zone_cols) != 1 or len(numeric_cols) != 1:
    raise ValueError("Expected exactly one zone column and one numeric column.")

zone_col = zone_cols[0]
numeric_col = numeric_cols[0]

# 3. ------------------------------------------------------------------ Group by zone column and sum the numeric column
grouped = transmission_lines_df.groupby(zone_col, as_index=False)[numeric_col].sum()

# 4. ----------------------------------------------------------- Replace the original DataFrame with the grouped result
transmission_lines_df = grouped

# 5. ------------------------------------------------------------------------------------------------------------- Done
print(transmission_lines_df)

   From -> to  Optimized capacity
0    BE -> FR         5212.888566
1    BE -> NL         4121.819454
2    DE -> BE         1349.999326
3    DE -> FR         4000.584940
4    DE -> NL         5682.486757
5    FR -> UK         1000.007702
6    UK -> BE         1349.999090
7    UK -> DE         1889.997636
8    UK -> FR         8479.714846
9    UK -> NL         2969.997804
10   UK -> UK         3713.846101


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Excluding intra-country internal connections.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [104]:
# 1. ------------------------------------------------------------ Identify the column containing values from zone names
zone_cols = []
for col in transmission_lines_df.columns:
    if transmission_lines_df[col].astype(str).str.contains('|'.join(zone_names)).any():
        zone_cols.append(col)

if not zone_cols:
    raise ValueError("No columns contain values from zone_names.")

zone_col = zone_cols[0]

# 2. -------------------------------------------------------------- Rename all columns except the selected one to 'NTC'
for col in transmission_lines_df.columns:
    if col != zone_col:
        transmission_lines_df = transmission_lines_df.rename(columns={col: 'NTC'})

# 3. ---------------------------------------------------- Check each row in the selected column for repeated zone names
def has_repeated_zones(text, zones):
    for zone in zones:
        if str(text).count(zone) >= 2:
            return True
    return False

# 4. -------------------------------------------------- Filter out rows with repeated zone_names in the selected column
transmission_lines_df = transmission_lines_df[
    ~transmission_lines_df[zone_col].apply(lambda x: has_repeated_zones(x, zone_names))
]

# 5. ------------------------------------------------------------------------------------------------------------- Done
print(transmission_lines_df)

  From -> to          NTC
0   BE -> FR  5212.888566
1   BE -> NL  4121.819454
2   DE -> BE  1349.999326
3   DE -> FR  4000.584940
4   DE -> NL  5682.486757
5   FR -> UK  1000.007702
6   UK -> BE  1349.999090
7   UK -> DE  1889.997636
8   UK -> FR  8479.714846
9   UK -> NL  2969.997804


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Creating the formated NTC time series.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [105]:
# 1. ------------------------------------------------------------------------------------------ Identify numeric column
numeric_cols = transmission_lines_df.select_dtypes(include="number").columns.tolist()
if not numeric_cols:
    raise ValueError("No numeric columns found in transmission_lines_df")
numeric_col = numeric_cols[0]  # choose the first numeric column

# 2. ---------------------------------------------------------------------------------------- Identify text/zone column
zone_cols = [
    col for col in transmission_lines_df.columns
    if transmission_lines_df[col].astype(str)
    .apply(lambda x: any(zone in x for zone in zone_names)).any()
]

if not zone_cols:
    raise ValueError("No zone/text columns found in transmission_lines_df")
zone_col = zone_cols[0]  # choose the first zone/text column

# 3. ------------------------------------------------------------------------------------------------- Process each row
for _, row in transmission_lines_df.iterrows():
    text_value = str(row[zone_col]).strip()  # e.g., "AA -> BB"
    numeric_value = row[numeric_col]

    if pd.isna(numeric_value):
        continue

# 3.1. -------------------------------------------------------------------------------- Split text_value into AA and BB
    if "->" not in text_value:
        continue  # skip if not in expected format

    parts = [part.strip() for part in text_value.split("->")]
    if len(parts) != 2:
        continue  # skip invalid format

    aa, bb = parts

# 3.2. ----------------------------------------------------- Prepare the two possible column names: original and mirror
    possible_columns = [f"{aa} -> {bb}", f"{bb} -> {aa}"]

    # Overwrite the numeric value in matching ntc_df columns
    for col in possible_columns:
        if col in ntc_df.columns:
            ntc_df[col] = numeric_value

# 4. ------------------------------------------------------------------------------------------------------------- Done
print(ntc_df)

                         index     BE -> FR     BE -> DE    BE -> UK  \
0    2030-01-01 00:00:00+00:00  5212.888566  1349.999326  1349.99909   
1    2030-01-01 01:00:00+00:00  5212.888566  1349.999326  1349.99909   
2    2030-01-01 02:00:00+00:00  5212.888566  1349.999326  1349.99909   
3    2030-01-01 03:00:00+00:00  5212.888566  1349.999326  1349.99909   
4    2030-01-01 04:00:00+00:00  5212.888566  1349.999326  1349.99909   
...                        ...          ...          ...         ...   
8779 2030-01-01 19:00:00+00:00  5212.888566  1349.999326  1349.99909   
8780 2030-01-01 20:00:00+00:00  5212.888566  1349.999326  1349.99909   
8781 2030-01-01 21:00:00+00:00  5212.888566  1349.999326  1349.99909   
8782 2030-01-01 22:00:00+00:00  5212.888566  1349.999326  1349.99909   
8783 2030-01-01 23:00:00+00:00  5212.888566  1349.999326  1349.99909   

         BE -> NL    DE -> FR     DE -> BE     DE -> NL     FR -> BE  \
0     4121.819454  4000.58494  1349.999326  5682.486757  5212.8

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
6.2.5. Formated NTC Data Files
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Storing the formatted and disaggregated NTC data into the designated directory within the Dispa-SET folder structure.
    <hr style="border: 0.5px solid skyblue;">
</div>

In [106]:
# 1. -------------------------------------------------------------------------------------- Ensure the directory exists
os.makedirs(ntc_pypsa_formated_data_folder_path, exist_ok=True)

# 2. ----------------------------------------------------------------------------------------- Define the CSV file path
csv_filename = f"{data_target_year}.csv"
csv_filepath = os.path.join(ntc_pypsa_formated_data_folder_path, csv_filename)

# 3. ---------------------------------------------------------------------------------------- Save the DataFrame to CSV
ntc_df.to_csv(csv_filepath, index=False)

# 4. ------------------------------------------------------------------------------------------------------------- Done
print(f"DataFrame successfully saved to: {csv_filepath}")

DataFrame successfully saved to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/NTC/1h/2030.csv


<div style="background-color: black;">
<hr style="border: 4px solid skyblue;">
</div>